# Regular RTH Stock Bars Example

This example subscribes to regular-hours stock bars with `useRTH=True` while leaving order routing permission independent with `outside_rth=True` on `SMART`.

In [ ]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

from ib_async import IB, util

# Required for sync ib.connect(...) inside Jupyter/IPython kernels.
util.startLoop()

repo_root = Path.cwd().resolve()
if repo_root.name == "examples":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import DEFAULT_LIVE_CONFIG, LiveTradingConfig
from execution import build_execution_instrument, request_underlying_realtime_bars
from orders import IBKRLimitOrderRouter
from strategies.live_mean_reversion import LiveMeanReversion


In [ ]:
SYMBOL = "META"
IB_HOST = "127.0.0.1"
IB_PORT = 4002
IB_CLIENT_ID = 111

raw = copy.deepcopy(DEFAULT_LIVE_CONFIG)
raw["symbol"] = SYMBOL
raw["model"]["enabled"] = False
raw["execution"]["market_data"] = {
    "exchange": "SMART",
    "barSize": 5,
    "whatToShow": "TRADES",
    "useRTH": True,
}
raw["execution"]["outside_rth"] = True
raw["execution"]["instrument"] = {
    "type": "stock",
    "exchange": "SMART",
    "currency": "USD",
    "limit_entry_offset_pct": 0.0005,
    "limit_exit_offset_pct": 0.0005,
}

config = LiveTradingConfig.from_dict(raw)
config.execution


In [ ]:
ib = IB()
if not ib.isConnected():
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT_ID)

stock, real_time_bars = request_underlying_realtime_bars(
    ib=ib,
    symbol=config.symbol,
    market_data_cfg=config.execution["market_data"],
)

execution_instrument = build_execution_instrument(ib, config.execution, config.symbol)
order_router = IBKRLimitOrderRouter(ib=ib, contract=stock)
algo = LiveMeanReversion(config=config, order_router=order_router, execution_instrument=execution_instrument)

print("Market data contract:", stock)
print("useRTH:", config.execution["market_data"]["useRTH"])
print("outside_rth orders:", config.execution["outside_rth"])


In [ ]:
# Attach when ready.
# real_time_bars.updateEvent += algo.on_bar

# Detach before rerunning setup.
# real_time_bars.updateEvent -= algo.on_bar
